# FINS5557 Week 3 — LLMs, Multimodal Models, and Finance Applications
**UNSW Business School | Hric & Lin, 2026**

---

This notebook provides hands-on implementations of the core concepts from Week 3 seminar notes. Each section maps directly to a seminar topic and can be run independently.

| § | Topic | Seminar Section | Key output |
|---|---|---|---|
| 1 | LSTM Architecture from Scratch | §1.1 — From LSTM to Transformer | Gate activation visualisation |
| 2 | Bitcoin Price Prediction (PyTorch LSTM) | §1.1 — Reference Chapter 7 | Price forecast vs actual |
| 3 | Sinusoidal Positional Encoding | §1.3 — Positional Encoding | Heatmap of PE vectors |
| 4 | Multi-Head Attention from Scratch | §1.4–1.5 — Attention Mechanism | Attention weight heatmaps |
| 5 | Vision Transformer: Image Patching | §3.1.1 — ViT [Dosovitskiy et al., 2021] | Patch grid, embedding matrix, [CLASS] attention map |
| 6 | CLIP: Cross-Modal Embeddings | §3.1.2 — CLIP [Radford et al., 2021] | Similarity matrix, zero-shot document classification |
| 7 | BLIP-2 Q-Former Architecture | §3.1.3 — BLIP-2 [Li et al., 2023] | Cross-attention weights, visual prefix, cost table |
| 8 | BloombergGPT Dataset Analysis | §2.2 — BloombergGPT | FinPile composition chart |
| 9 | LoRA Parameter Efficiency | §2.4 — LoRA Adaptation | Compression ratio analysis |
| 10 | Mixture of Experts Routing | §2.5 — MoE Architecture | Expert load distribution |
| 11 | Gemini Multimodal + VQA Case Study | §3.2–3.3 — Gemini, VQA | APRA CPS 220 report; 4 financial VQA scenarios |
| 12 | APRA Capital Ratio Dashboard | §3.3 — Case Study | Capital adequacy dashboard |

**Runtime recommendation:** GPU (T4 or better) for §2 Bitcoin LSTM training. All other sections run on CPU.

**API note:** §11 includes both a live Gemini API path and a fully self-contained mock path.

In [ ]:
# Setup — install required packages (run once at session start)
# Uncomment the lines you need
# !pip install torch torchvision --quiet
# !pip install google-generativeai --quiet
# !pip install requests pandas numpy matplotlib seaborn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.special import softmax
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
NAVY   = '#003366'
RED    = '#C8102E'
ORANGE = '#D4820A'
GREEN  = '#1A7A4A'
LIGHT  = '#F4F6F9'

print('Environment ready.')

---
## §1 — LSTM Architecture from Scratch

**Seminar reference:** Chapter 7, Language Analytics for Finance and FinTech [Hric & Lin, 2026]

This section implements a complete LSTM cell in pure NumPy, tracking all gate activations. This establishes the baseline before the transformer architecture is derived in §3–4.

**LSTM gate equations:**
$$f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i) \quad \text{(input gate)}$$
$$\tilde{c}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c) \quad \text{(candidate values)}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \quad \text{(cell state)}$$
$$h_t = o_t \odot \tanh(c_t) \quad \text{(hidden state)}$$

In [ ]:
# §1: LSTM Cell — full NumPy implementation (from Chapter 7 [Hric & Lin, 2026])

class LSTMCell:
    """LSTM cell with explicit gate tracking for pedagogical inspection."""

    def __init__(self, input_size: int, hidden_size: int, seed: int = 42):
        rng = np.random.default_rng(seed)
        scale = 0.1
        # Forget, Input, Candidate, Output gate weights
        for gate in ['f', 'i', 'c', 'o']:
            setattr(self, f'W{gate}', rng.normal(0, scale, (hidden_size, input_size)))
            setattr(self, f'U{gate}', rng.normal(0, scale, (hidden_size, hidden_size)))
            setattr(self, f'b{gate}', np.zeros((hidden_size, 1)))
        self.history = {'f': [], 'i': [], 'o': [], 'c': [], 'h': []}

    def sigmoid(self, x): return 1 / (1 + np.exp(-np.clip(x, -15, 15)))

    def forward(self, x_t, h_prev, c_prev):
        x = x_t.reshape(-1, 1)
        h = h_prev.reshape(-1, 1)
        c = c_prev.reshape(-1, 1)

        f_t = self.sigmoid(self.Wf @ x + self.Uf @ h + self.bf)   # forget
        i_t = self.sigmoid(self.Wi @ x + self.Ui @ h + self.bi)   # input
        c_tilde = np.tanh(self.Wc @ x + self.Uc @ h + self.bc)    # candidate
        c_t = f_t * c + i_t * c_tilde                              # cell state
        o_t = self.sigmoid(self.Wo @ x + self.Uo @ h + self.bo)   # output
        h_t = o_t * np.tanh(c_t)                                   # hidden state

        self.history['f'].append(float(f_t.mean()))
        self.history['i'].append(float(i_t.mean()))
        self.history['o'].append(float(o_t.mean()))
        self.history['c'].append(float(c_t.mean()))
        self.history['h'].append(float(h_t.mean()))
        return h_t.flatten(), c_t.flatten()


# Simulate a macro shock scenario: RBA cash rate signal series
# T=1..24 months: stable tightening cycle; T=25 sudden commodity price shock
np.random.seed(42)
T = 48
clr_series = np.concatenate([
    np.linspace(0.1, 0.8, 24),          # rising tightening cycle
    np.linspace(0.8, 0.3, 12),          # CPI drops — tightening ends
    0.9 + 0.05 * np.random.randn(12),   # commodity shock
])

lstm = LSTMCell(input_size=1, hidden_size=4)
h = np.zeros(4)
c = np.zeros(4)
for t in range(T):
    x = np.array([clr_series[t]])
    h, c = lstm.forward(x, h, c)

# Plot: 3-panel gate activation analysis
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
t_axis = np.arange(1, T + 1)

axes[0].plot(t_axis, clr_series, color=NAVY, lw=2)
axes[0].axvline(24, color=RED, linestyle='--', alpha=0.7, label='CPI drop (tightening ends)')
axes[0].axvline(36, color=ORANGE, linestyle='--', alpha=0.7, label='Commodity shock')
axes[0].set_ylabel('CLR signal', fontsize=10)
axes[0].set_title('LSTM Gate Dynamics — RBA Cash Rate Scenario', fontsize=12, color=NAVY)
axes[0].legend(fontsize=8)

axes[1].plot(t_axis, lstm.history['f'], color=RED, lw=2, label='Forget gate')
axes[1].plot(t_axis, lstm.history['i'], color=GREEN, lw=2, label='Input gate')
axes[1].plot(t_axis, lstm.history['o'], color=ORANGE, lw=2, label='Output gate')
axes[1].set_ylabel('Gate activation [0,1]', fontsize=10)
axes[1].legend(fontsize=8)

axes[2].plot(t_axis, lstm.history['c'], color=NAVY, lw=2, label='Cell state')
axes[2].plot(t_axis, lstm.history['h'], color='purple', lw=2, linestyle='--', label='Hidden state')
axes[2].set_ylabel('State magnitude', fontsize=10)
axes[2].set_xlabel('Month', fontsize=10)
axes[2].legend(fontsize=8)

for ax in axes:
    ax.axvline(24, color=RED, linestyle='--', alpha=0.4)
    ax.axvline(36, color=ORANGE, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('lstm_gate_dynamics.png', bbox_inches='tight')
plt.show()

print(f'Gate values at CPI drop (t=24): forget={lstm.history["f"][23]:.3f}, input={lstm.history["i"][23]:.3f}')
print(f'Gate values at commodity shock (t=36): forget={lstm.history["f"][35]:.3f}, input={lstm.history["i"][35]:.3f}')
print('\nKey insight: forget gate collapses at CPI drop → LSTM discards tightening memory.')
print('Input gate spikes at commodity shock → new dominant signal written to cell state.')

---
## §2 — Bitcoin Price Prediction (PyTorch LSTM)

**Seminar reference:** Chapter 7, §7.3 Structured Data — Crypto Quant Trading [Hric & Lin, 2026]

Production PyTorch LSTM with 2 layers, dropout regularisation, and MinMax scaling. Predicts next-day Bitcoin price using a 30-day lookback window.

> **LSTM advantage:** captures non-linear regime changes (bull/bear transitions) that simple moving average approaches miss — directly analogous to the RBA rate forecasting scenario in §1.

In [ ]:
# §2: Bitcoin LSTM — PyTorch implementation (from Chapter 7 [Hric & Lin, 2026])
# Requires: torch, requests; uses CryptoCompare public API (no key required)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from sklearn.preprocessing import MinMaxScaler
    TORCH_AVAILABLE = True
    print(f'PyTorch {torch.__version__} available. Device: {"GPU" if torch.cuda.is_available() else "CPU"}')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not installed — running in demo mode with synthetic Bitcoin data.')

import requests

def fetch_bitcoin_data(days: int = 365) -> pd.DataFrame:
    """Fetch Bitcoin daily close prices from CryptoCompare public API."""
    url = 'https://min-api.cryptocompare.com/data/v2/histoday'
    params = {'fsym': 'BTC', 'tsym': 'USD', 'limit': days, 'aggregate': 1}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()['Data']['Data']
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['time'], unit='s')
        return df[['date', 'close']].rename(columns={'close': 'price'}).sort_values('date').reset_index(drop=True)
    except Exception as e:
        print(f'API unavailable ({e}). Using synthetic data.')
        dates = pd.date_range('2024-01-01', periods=days)
        prices = 42000 + np.cumsum(np.random.randn(days) * 800)
        return pd.DataFrame({'date': dates, 'price': np.abs(prices)})

df_btc = fetch_bitcoin_data(365)
print(f'Fetched {len(df_btc)} days | Price range: ${df_btc.price.min():,.0f} – ${df_btc.price.max():,.0f}')
print(f'Date range: {df_btc.date.min().date()} to {df_btc.date.max().date()}')

# Data preparation
SEQ_LEN   = 30
TRAIN_PCT = 0.80

scaler = MinMaxScaler(feature_range=(0, 1))
prices_scaled = scaler.fit_transform(df_btc[['price']]).flatten()
split = int(len(prices_scaled) * TRAIN_PCT)

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)

X_train, y_train = make_sequences(prices_scaled[:split], SEQ_LEN)
X_test,  y_test  = make_sequences(prices_scaled[split:], SEQ_LEN)

if TORCH_AVAILABLE:
    class BitcoinDataset(Dataset):
        def __init__(self, X, y):
            self.X = torch.FloatTensor(X).unsqueeze(-1)
            self.y = torch.FloatTensor(y).unsqueeze(-1)
        def __len__(self): return len(self.X)
        def __getitem__(self, i): return self.X[i], self.y[i]

    class BitcoinLSTM(nn.Module):
        """2-layer LSTM with dropout for Bitcoin price prediction."""
        def __init__(self, hidden=50, layers=2, dropout=0.2):
            super().__init__()
            self.lstm   = nn.LSTM(1, hidden, layers, batch_first=True, dropout=dropout)
            self.drop   = nn.Dropout(dropout)
            self.linear = nn.Linear(hidden, 1)
        def forward(self, x):
            h0 = torch.zeros(2, x.size(0), 50, device=x.device)
            c0 = torch.zeros(2, x.size(0), 50, device=x.device)
            out, _ = self.lstm(x, (h0, c0))
            return self.linear(self.drop(out[:, -1, :]))

    train_loader = DataLoader(BitcoinDataset(X_train, y_train), batch_size=32, shuffle=True)
    test_loader  = DataLoader(BitcoinDataset(X_test, y_test),  batch_size=32, shuffle=False)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model     = BitcoinLSTM().to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5)

    train_losses, val_losses = [], []
    EPOCHS = 60
    for epoch in range(EPOCHS):
        model.train()
        tloss = 0.0
        for bx, by in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(bx.to(device)), by.to(device))
            loss.backward()
            optimizer.step()
            tloss += loss.item()
        tloss /= len(train_loader)

        model.eval()
        with torch.no_grad():
            vloss = sum(criterion(model(bx.to(device)), by.to(device)).item()
                        for bx, by in test_loader) / len(test_loader)
        scheduler.step(vloss)
        train_losses.append(tloss); val_losses.append(vloss)
        if (epoch + 1) % 15 == 0:
            print(f'Epoch {epoch+1:3d}/{EPOCHS} | Train Loss: {tloss:.5f} | Val Loss: {vloss:.5f}')

    model.eval()
    with torch.no_grad():
        preds = []
        for bx, _ in test_loader:
            preds.extend(model(bx.to(device)).cpu().numpy().flatten())

    preds_actual = scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()
    actuals      = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    rmse = np.sqrt(np.mean((preds_actual - actuals) ** 2))
    print(f'\nTest RMSE: ${rmse:,.0f}')

    # Predict next day
    last_seq = torch.FloatTensor(prices_scaled[-SEQ_LEN:]).unsqueeze(0).unsqueeze(-1).to(device)
    with torch.no_grad():
        pred_next = scaler.inverse_transform(model(last_seq).cpu().numpy())[0, 0]
    current = df_btc['price'].iloc[-1]
    print(f'Current BTC price:   ${current:>12,.2f}')
    print(f'Predicted next day:  ${pred_next:>12,.2f}')
    print(f'Predicted change:    ${pred_next - current:>+12,.2f} ({(pred_next/current - 1)*100:+.2f}%)')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(actuals[-60:], color=NAVY, lw=2, label='Actual')
    axes[0].plot(preds_actual[-60:], color=RED, lw=2, linestyle='--', label='Predicted')
    axes[0].set_title('Bitcoin Price: Actual vs Predicted (last 60 test days)', color=NAVY)
    axes[0].set_ylabel('Price (USD)'); axes[0].legend()

    axes[1].plot(train_losses, color=NAVY, label='Train loss')
    axes[1].plot(val_losses,   color=RED,  label='Val loss')
    axes[1].set_title('Training Convergence', color=NAVY)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss'); axes[1].legend()
    plt.tight_layout(); plt.show()
else:
    print('Demo mode: install PyTorch and rerun for full Bitcoin LSTM training.')

---
## §3 — Sinusoidal Positional Encoding

**Seminar reference:** Week 3, §1.3 — Positional Encoding

Self-attention is **permutation-invariant** — processing tokens in any order produces the same output. Positional encoding injects position information by adding a deterministic signal to each token embedding.

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad
PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

Each position produces a unique fingerprint across $d_{\text{model}}$ dimensions. Nearby positions have similar (high cosine similarity) fingerprints — proximity is encoded geometrically.

In [ ]:
# §3: Sinusoidal Positional Encoding — visualisation

def sinusoidal_pe(max_len: int, d_model: int) -> np.ndarray:
    """Compute sinusoidal positional encoding matrix (Vaswani et al., 2017)."""
    PE = np.zeros((max_len, d_model))
    positions = np.arange(max_len).reshape(-1, 1)          # (T, 1)
    div_term  = 10000 ** (2 * np.arange(d_model // 2) / d_model)  # (d/2,)
    PE[:, 0::2] = np.sin(positions / div_term)             # even dims
    PE[:, 1::2] = np.cos(positions / div_term[:d_model//2])  # odd dims
    return PE

MAX_LEN = 100
D_MODEL = 64
PE = sinusoidal_pe(MAX_LEN, D_MODEL)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: PE heatmap
im = axes[0].imshow(PE[:50, :32], aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xlabel('Embedding dimension'); axes[0].set_ylabel('Token position')
axes[0].set_title('Positional Encoding Heatmap\n(first 50 positions × 32 dims)', color=NAVY)
plt.colorbar(im, ax=axes[0])

# Panel 2: Specific dimension waveforms
for dim, col in [(0, NAVY), (4, RED), (8, GREEN), (16, ORANGE)]:
    axes[1].plot(PE[:, dim], label=f'dim {dim}', color=col, lw=1.5)
axes[1].set_xlabel('Token position'); axes[1].set_ylabel('PE value')
axes[1].set_title('PE Waveforms — Lower dims = low freq\n(long-range patterns)', color=NAVY)
axes[1].legend(fontsize=8)

# Panel 3: Cosine similarity between positions (proximity encoding)
n_pos = 30
sim_matrix = np.zeros((n_pos, n_pos))
for a in range(n_pos):
    for b in range(n_pos):
        sim_matrix[a, b] = np.dot(PE[a], PE[b]) / (np.linalg.norm(PE[a]) * np.linalg.norm(PE[b]))
im2 = axes[2].imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
axes[2].set_xlabel('Position A'); axes[2].set_ylabel('Position B')
axes[2].set_title('Cosine Similarity Between Positions\n(nearby positions ≈ similar PE)', color=NAVY)
plt.colorbar(im2, ax=axes[2])

plt.tight_layout(); plt.show()

# Demonstrate relative position property
pos5  = PE[5]
pos10 = PE[10]
pos25 = PE[25]
print(f'cosine(pos_5, pos_6)  = {np.dot(PE[5], PE[6])  / (np.linalg.norm(PE[5])  * np.linalg.norm(PE[6])):.4f}  (adjacent)')
print(f'cosine(pos_5, pos_25) = {np.dot(PE[5], PE[25]) / (np.linalg.norm(PE[5])  * np.linalg.norm(PE[25])):.4f}  (20 positions apart)')
print(f'cosine(pos_5, pos_55) = {np.dot(PE[5], PE[55]) / (np.linalg.norm(PE[5])  * np.linalg.norm(PE[55])):.4f}  (50 positions apart)')
print('\nKey insight: similarity decays with distance → proximity is geometrically encoded.')

---
## §4 — Multi-Head Attention from Scratch

**Seminar reference:** Week 3, §1.4–1.5 — Scaled Dot-Product Attention and Multi-Head Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H)W^O$$

**Finance scenario:** Bond indenture clause cross-reference. Token "unless" must identify token "§12.1" as the carve-out condition across a 12-token excerpt — demonstrated below with two parallel attention heads.

In [ ]:
# §4: Multi-Head Self-Attention — NumPy implementation with finance scenarios

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention (Vaswani et al., 2017)."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)          # (T, T)
    if mask is not None:
        scores = scores + mask                # causal masking: -inf for future tokens
    weights = softmax(scores, axis=-1)        # row-wise softmax
    output  = weights @ V
    return output, weights

def multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads, mask=None):
    """Multi-head attention with H parallel heads."""
    T, d_model = X.shape
    d_k = d_model // num_heads
    head_outputs, all_weights = [], []
    for h in range(num_heads):
        Q_h = X @ W_Q[h]   # (T, d_k)
        K_h = X @ W_K[h]
        V_h = X @ W_V[h]
        out_h, w_h = scaled_dot_product_attention(Q_h, K_h, V_h, mask)
        head_outputs.append(out_h)
        all_weights.append(w_h)
    concat = np.concatenate(head_outputs, axis=-1)   # (T, d_model)
    return concat @ W_O, all_weights


# --- Scenario 1: Bond Indenture Cross-Reference ---
# Tokens: "the Issuer shall not unless [See §12.1] pledge any assets"
tokens_bond = ['the', 'Issuer', 'shall', 'not', 'unless', '§12.1', 'pledge', 'any', 'assets']
T1 = len(tokens_bond)
d_model, num_heads = 8, 2
d_k = d_model // num_heads

rng = np.random.default_rng(0)
# Craft embeddings: "unless" (idx=4) and "§12.1" (idx=5) share a high dot-product dimension
X_bond = rng.normal(0, 0.5, (T1, d_model))
X_bond[4, 0] =  2.5  # unless: strong signal on dim 0
X_bond[5, 0] =  2.5  # §12.1: matching signal
X_bond[4, 1] =  1.8  # unless: secondary signal
X_bond[5, 2] = -0.5  # §12.1: orthogonal secondary

W_Qs = [rng.normal(0, 0.3, (d_model, d_k)) for _ in range(num_heads)]
W_Ks = [rng.normal(0, 0.3, (d_model, d_k)) for _ in range(num_heads)]
W_Vs = [rng.normal(0, 0.3, (d_model, d_k)) for _ in range(num_heads)]
W_O  =  rng.normal(0, 0.3, (d_model, d_model))

# Amplify head 0 to pick up the unless→§12.1 relationship
W_Qs[0][0, 0] = 3.0; W_Ks[0][0, 0] = 3.0

_, weights_bond = multi_head_attention(X_bond, W_Qs, W_Ks, W_Vs, W_O, num_heads)


# --- Scenario 2: Earnings call sentiment — "provisions" disambiguation ---
# "Net interest income increased while provisions for credit loss rose sharply"
tokens_earn = ['Net', 'interest', 'income', 'increased', 'while', 'provisions', 'for', 'credit', 'loss', 'rose']
T2 = len(tokens_earn)
X_earn = rng.normal(0, 0.5, (T2, d_model))
# "provisions" → attends strongly to "credit" and "loss" (negative cluster)
X_earn[5, 3] =  2.0; X_earn[7, 3] =  2.0; X_earn[8, 3] =  1.8  # provisions↔credit↔loss
# "increased" → attends to "income" (positive cluster)
X_earn[3, 4] =  2.0; X_earn[2, 4] =  2.0
W_Qs2 = [rng.normal(0, 0.3, (d_model, d_k)) for _ in range(num_heads)]
W_Ks2 = [rng.normal(0, 0.3, (d_model, d_k)) for _ in range(num_heads)]
W_Vs2 = [rng.normal(0, 0.3, (d_model, d_k)) for _ in range(num_heads)]
W_Qs2[0][3, 0] = 3.0; W_Ks2[0][3, 0] = 3.0
W_Qs2[1][4, 0] = 3.0; W_Ks2[1][4, 0] = 3.0
_, weights_earn = multi_head_attention(X_earn, W_Qs2, W_Ks2, W_Vs2, W_O, num_heads)

# Plot attention heatmaps
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
scenarios = [
    (tokens_bond, weights_bond, 'Bond Indenture — Head 1\n"unless" → §12.1 cross-reference', 0, 0),
    (tokens_bond, weights_bond, 'Bond Indenture — Head 2\nSyntactic structure', 1, 0),
    (tokens_earn, weights_earn, 'Earnings Call — Head 1\n"provisions" → credit/loss cluster', 0, 1),
    (tokens_earn, weights_earn, 'Earnings Call — Head 2\n"increased" → income cluster', 1, 1),
]
for head_idx, (tokens, weights, title, col, row) in enumerate(scenarios):
    ax_row  = head_idx // 2
    ax_col  = head_idx  % 2
    head_w  = weights[head_idx % 2]
    im = axes[ax_row, ax_col].imshow(head_w, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    axes[ax_row, ax_col].set_xticks(range(len(tokens)))
    axes[ax_row, ax_col].set_yticks(range(len(tokens)))
    axes[ax_row, ax_col].set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    axes[ax_row, ax_col].set_yticklabels(tokens, fontsize=8)
    axes[ax_row, ax_col].set_title(title, color=NAVY, fontsize=10)
    plt.colorbar(im, ax=axes[ax_row, ax_col])

plt.suptitle('Multi-Head Self-Attention — Finance Scenarios\n(attention weight: row token attending to column token)',
             fontsize=11, color=NAVY, y=1.01)
plt.tight_layout(); plt.show()

# Report "unless" → §12.1 weight
unless_idx = tokens_bond.index('unless')
sec_idx    = tokens_bond.index('§12.1')
w1 = weights_bond[0][unless_idx, sec_idx]
w2 = weights_bond[1][unless_idx, sec_idx]
print(f'Head 1 — attention("unless" → "§12.1"): {w1:.3f}')
print(f'Head 2 — attention("unless" → "§12.1"): {w2:.3f}')
print('\nRNN gradient at step k=400 (for comparison): 0.9^400 ≈', round(0.9**400, 20))

---
## §5 — Vision Transformer: Image Patching and Patch Embeddings

**Seminar reference:** Week 3, §3.1.1 — Vision Transformer [Dosovitskiy et al., 2021]

The ViT converts a 2D image into a 1D sequence of patch token embeddings, enabling a standard
transformer encoder to process images with the same self-attention used for text:

$$\mathbf{z}_0 = [\mathbf{x}_{\text{class}}; \, \mathbf{x}_p^1 \mathbf{E}; \ldots; \mathbf{x}_p^N \mathbf{E}] + \mathbf{E}_{\text{pos}} \tag{3.1}$$

where $\mathbf{x}_p^i$ is the flattened pixel vector of patch $i$, $\mathbf{E}$ is the learned
linear projection, and $\mathbf{E}_{\text{pos}}$ encodes spatial structure. The prepended `[CLASS]`
token accumulates a global image-level representation, identical in role to BERT's `[CLS]`.

**Finance context:** ViT-based encoders locate CET1 capital ratios and provision tables directly
from scanned APRA regulatory filings — without a prior OCR pass. Spatial attention reads column
headers and cell boundaries geometrically.

In [ ]:
# §5: Vision Transformer (ViT) — Image Patching and Patch Embeddings
# Implements Equation 3.1. Finance: APRA filing page -> patch tokens -> [CLASS] attention map.

from scipy.special import softmax as scipy_softmax

IMG_SIZE   = 64
PATCH_SIZE = 8
N_PATCHES  = (IMG_SIZE // PATCH_SIZE) ** 2   # 64 patches
D_PATCH    = PATCH_SIZE ** 2                  # 64 dims per flat patch
D_VIT      = 32                               # hidden dim (real ViT-B/16: 768)

rng5 = np.random.default_rng(1)

# 1. Synthetic APRA filing page (table-like structure)
page_vit = np.ones((IMG_SIZE, IMG_SIZE), dtype=np.float32) * 0.95
for i in range(0, IMG_SIZE, IMG_SIZE // 4):
    page_vit[i:i+2, :] = 0.1            # horizontal table rules
for j in range(0, IMG_SIZE, IMG_SIZE // 4):
    page_vit[:, j:j+2] = 0.1            # vertical table rules
for r in range(4, IMG_SIZE, 16):
    for c in range(4, IMG_SIZE, 16):
        page_vit[r:r+3, c:c+8] = 0.2 + rng5.random() * 0.3

# 2. Extract and flatten patches
def extract_patches_vit(image, psz):
    H, W = image.shape
    return np.array([
        image[i*psz:(i+1)*psz, j*psz:(j+1)*psz].flatten()
        for i in range(H // psz) for j in range(W // psz)
    ])

flat_vit = extract_patches_vit(page_vit, PATCH_SIZE)   # (64, 64)

# 3. Linear projection E and [CLASS] token
E_vit      = rng5.normal(0, 1 / np.sqrt(D_PATCH), (D_PATCH, D_VIT))
emb_vit    = flat_vit @ E_vit                           # (64, 32)
cls_vit    = np.zeros((1, D_VIT))

# 4. Sinusoidal positional embeddings
def pe_fn(max_len, d):
    PE  = np.zeros((max_len, d))
    pos = np.arange(max_len)[:, None]
    div = 10000 ** (2 * np.arange(d // 2) / d)
    PE[:, 0::2] = np.sin(pos / div)
    PE[:, 1::2] = np.cos(pos / div)
    return PE

E_pos_vit = pe_fn(N_PATCHES + 1, D_VIT)   # (65, 32)

# 5. Equation 3.1 — z_0
z_0_vit = np.vstack([cls_vit, emb_vit]) + E_pos_vit   # (65, 32)

print(f'z_0 shape: {z_0_vit.shape}  ([CLASS] + {N_PATCHES} patch embeddings, D={D_VIT})')
print(f'[CLASS] z_0[0,:6]:   {z_0_vit[0, :6].round(3)}')
print(f'Patch 1 z_0[1,:6]:   {z_0_vit[1, :6].round(3)}')

# 6. Single-head self-attention (demonstration)
W_Q5 = rng5.normal(0, 0.2, (D_VIT, D_VIT // 2))
W_K5 = rng5.normal(0, 0.2, (D_VIT, D_VIT // 2))
scores_vit  = (z_0_vit @ W_Q5) @ (z_0_vit @ W_K5).T / np.sqrt(D_VIT // 2)
attn_vit    = scipy_softmax(scores_vit, axis=-1)
top5_patches = np.argsort(attn_vit[0, 1:])[::-1][:5] + 1
print(f'[CLASS] attends most to patches: {top5_patches}  (table border / numeric cells)')

# 7. Four-panel visualisation
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(page_vit, cmap='gray', vmin=0, vmax=1)
for i in range(0, IMG_SIZE + 1, PATCH_SIZE):
    axes[0].axhline(i - 0.5, color='red', lw=0.8, alpha=0.7)
    axes[0].axvline(i - 0.5, color='red', lw=0.8, alpha=0.7)
axes[0].set_title(f'Synthetic APRA Filing Page\n({IMG_SIZE}x{IMG_SIZE} px, patch grid in red)', color=NAVY)
axes[0].set_xlabel('px col'); axes[0].set_ylabel('px row')

axes[1].set_axis_off()
axes[1].set_title(f'First 16 Patches ({PATCH_SIZE}x{PATCH_SIZE} px each)', color=NAVY)
for idx in range(16):
    ri, ci = idx // 4, idx % 4
    ax_in = axes[1].inset_axes([ci/4+0.01, 1-(ri+1)/4+0.01, 0.23, 0.22])
    ax_in.imshow(flat_vit[idx].reshape(PATCH_SIZE, PATCH_SIZE), cmap='gray', vmin=0, vmax=1)
    ax_in.set_title(f'P{idx+1}', fontsize=6, pad=1); ax_in.set_xticks([]); ax_in.set_yticks([])

imc = axes[2].imshow(emb_vit, aspect='auto', cmap='RdBu_r')
axes[2].set_title(f'Patch Embedding Matrix\n({N_PATCHES} patches x D={D_VIT})', color=NAVY)
axes[2].set_xlabel('Embedding dim'); axes[2].set_ylabel('Patch index')
plt.colorbar(imc, ax=axes[2])

cls_map = attn_vit[0, 1:].reshape(IMG_SIZE // PATCH_SIZE, IMG_SIZE // PATCH_SIZE)
imd = axes[3].imshow(cls_map, cmap='YlOrRd', vmin=0)
axes[3].set_title('[CLASS] Attention Map\n(which patches inform global representation?)', color=NAVY)
axes[3].set_xlabel('Patch col'); axes[3].set_ylabel('Patch row')
plt.colorbar(imd, ax=axes[3])
mp = np.unravel_index(cls_map.argmax(), cls_map.shape)
axes[3].text(mp[1], mp[0], '*', ha='center', va='center', fontsize=16, color='white', fontweight='bold')

plt.suptitle('Vision Transformer (Eq 3.1) — APRA Capital Filing\n'
             'Spatial attention reads table structure; no OCR pre-pass required',
             fontsize=11, color=NAVY, y=1.02)
plt.tight_layout(); plt.show()

print('\nScaling: real ViT-B/16 (224x224, 16x16 patches) -> 196 tokens/page')
print(f'200-page APRA report: 196 x 200 = 39,200 tokens')
print(f'Gemini 1.5 Pro 1M context: ~25 full 200-page reports simultaneously')

---
## §6 — CLIP: Contrastive Cross-Modal Embeddings

**Seminar reference:** Week 3, §3.1.2 — CLIP [Radford et al., 2021]

CLIP learns a **shared embedding space** in which semantically related images and text are
geometrically proximate, enabling cross-modal similarity computation without labelled training data:

$$\text{sim}(I, T) = \frac{\phi(I) \cdot \psi(T)}{\|\phi(I)\| \, \|\psi(T)\|} \tag{3.2}$$

$$\mathcal{L}_{\text{CLIP}} = -\frac{1}{N} \sum_{i=1}^{N} \Bigl[ \log \frac{e^{\phi(I_i)\cdot\psi(T_i)/\tau}}{\sum_j e^{\phi(I_i)\cdot\psi(T_j)/\tau}} + \log \frac{e^{\psi(T_i)\cdot\phi(I_i)/\tau}}{\sum_j e^{\psi(T_j)\cdot\phi(I_i)/\tau}} \Bigr] \tag{3.3}$$

Training on 400 million (image, caption) pairs pulls matched pairs toward high cosine similarity
and pushes unmatched pairs toward zero.

**Finance applications:** cross-modal document retrieval (text query → matching page images, no OCR),
zero-shot page classification (balance sheet / CET1 table / auditor report from text descriptions),
and regulatory change triage (SR 11-7 vs SR 26-2 similarity scoring).

In [ ]:
# §6: CLIP — Contrastive Language-Image Pre-training [Radford et al., 2021]
# Implements Equations 3.2 and 3.3. Finance: zero-shot financial document classification.

N_CLIP = 6
D_CLIP = 16    # shared embedding dim (real CLIP: 512)
LABELS = ['Balance\nSheet','Revenue\nChart','Auditor\nReport',
          'Exec\nBio','CET1\nTable','Risk\nFactors']
rng6 = np.random.default_rng(42)

# Simulate learned image encoder phi and text encoder psi
# Each document type has a unique signature in one pair of dimensions
phi_r = rng6.normal(0, 1, (N_CLIP, D_CLIP))
psi_r = rng6.normal(0, 1, (N_CLIP, D_CLIP))
for i in range(N_CLIP):
    phi_r[i, i*2:(i+1)*2] += 2.5
    psi_r[i, i*2:(i+1)*2] += 2.5

# Equation 3.2: L2-normalise then compute cosine similarity matrix
phi_clip = phi_r / np.linalg.norm(phi_r, axis=1, keepdims=True)
psi_clip = psi_r / np.linalg.norm(psi_r, axis=1, keepdims=True)
S_clip   = phi_clip @ psi_clip.T    # (N, N)

print('CLIP Shared Embedding Space')
print(f'  D_embed={D_CLIP}, N={N_CLIP} document types')
print(f'  Diagonal (matched pairs):      {np.diag(S_clip).round(3)}')
print(f'  Mean off-diagonal (unmatched): {(S_clip.sum()-np.trace(S_clip))/(N_CLIP*(N_CLIP-1)):.3f}')

# Equation 3.3: symmetric InfoNCE contrastive loss
tau_clip = 0.07

def infonce(S, tau):
    N = len(S)
    Ss = S / tau
    l_i2t = -np.mean([Ss[i,i] - np.log(np.sum(np.exp(Ss[i] - Ss[i].max()))) for i in range(N)])
    l_t2i = -np.mean([Ss[i,i] - np.log(np.sum(np.exp(Ss[:,i] - Ss[:,i].max()))) for i in range(N)])
    return (l_i2t + l_t2i) / 2

loss_clip = infonce(S_clip, tau_clip)
print(f'\nEquation 3.3 -- InfoNCE loss: {loss_clip:.4f}  (tau={tau_clip})')

# Zero-shot page classification (no labelled training data)
psi_d = rng6.normal(0, 0.5, (N_CLIP, D_CLIP))
for i in range(N_CLIP):
    psi_d[i, i*2:(i+1)*2] += 2.0
psi_d /= np.linalg.norm(psi_d, axis=1, keepdims=True)
pred = (phi_clip @ psi_d.T).argmax(axis=1)
acc  = float(np.mean(pred == np.arange(N_CLIP)))
flat = [l.replace('\n', ' ') for l in LABELS]
print(f'\nZero-shot accuracy: {acc:.0%}  (no labelled data required)')
for i in range(N_CLIP):
    mark = 'OK' if pred[i] == i else 'XX'
    print(f'  {mark}  {flat[i]:<18} -> {flat[pred[i]]}')

# Cross-modal retrieval
q = rng6.normal(0, 0.5, (1, D_CLIP)); q[0, 8:10] += 2.5
q /= np.linalg.norm(q)
sims_r = (phi_clip @ q.T).flatten()
print('\nCross-modal retrieval: text="capital ratio regulatory disclosure table"')
for rank, idx in enumerate(np.argsort(sims_r)[::-1]):
    print(f'  {rank+1}. {flat[idx]:<18}  sim={sims_r[idx]:.3f}')

# Three-panel visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im_a = axes[0].imshow(S_clip, cmap='RdYlGn', vmin=-0.3, vmax=1.0, aspect='auto')
axes[0].set_xticks(range(N_CLIP)); axes[0].set_yticks(range(N_CLIP))
axes[0].set_xticklabels(flat, fontsize=7, rotation=45, ha='right')
axes[0].set_yticklabels(flat, fontsize=7)
axes[0].set_xlabel('Text embedding psi(T)'); axes[0].set_ylabel('Image embedding phi(I)')
axes[0].set_title('Cosine Similarity Matrix (Eq 3.2)\nDiagonal = matched pairs', color=NAVY)
plt.colorbar(im_a, ax=axes[0])
for i in range(N_CLIP):
    axes[0].add_patch(plt.Rectangle((i-.5,i-.5),1,1,fill=False,edgecolor='white',lw=2))

zs_probs = scipy_softmax((phi_clip @ psi_d.T) / tau_clip, axis=1)
for i in range(3):
    axes[1].bar(np.arange(N_CLIP)+i*0.27, zs_probs[i], width=0.26,
                label=flat[i], color=[NAVY,RED,GREEN][i], alpha=0.85)
axes[1].axhline(1/N_CLIP, color='grey', linestyle='--', lw=1, label=f'Uniform 1/{N_CLIP}')
axes[1].set_xticks(np.arange(N_CLIP)+0.27)
axes[1].set_xticklabels(flat, fontsize=7, rotation=30, ha='right')
axes[1].set_ylabel('Softmax prob')
axes[1].set_title('Zero-Shot Classification Probs\n(3 documents, no labelled data)', color=NAVY)
axes[1].legend(fontsize=7)

from numpy.linalg import svd as _svd
av = np.vstack([phi_clip, psi_clip]); mu = av.mean(0)
_, _, Vt = _svd(av - mu, full_matrices=False)
p2d = (av - mu) @ Vt[:2].T
c6  = [NAVY, RED, GREEN, ORANGE, 'purple', '#8B4513']
for i in range(N_CLIP):
    axes[2].scatter(*p2d[i],         s=120, color=c6[i], marker='s', zorder=5)
    axes[2].scatter(*p2d[N_CLIP+i],  s=120, color=c6[i], marker='o', zorder=5)
    axes[2].plot([p2d[i,0],p2d[N_CLIP+i,0]],[p2d[i,1],p2d[N_CLIP+i,1]],
                color=c6[i], lw=1, linestyle='--', alpha=0.6)
    axes[2].annotate(flat[i], p2d[i], fontsize=6.5, ha='center', va='bottom')
sq = plt.scatter([],[],s=80,marker='s',color='grey',label='Image phi(I)')
ci = plt.scatter([],[],s=80,marker='o',color='grey',label='Text psi(T)')
axes[2].legend(handles=[sq,ci],fontsize=8)
axes[2].set_xlabel('PC 1'); axes[2].set_ylabel('PC 2')
axes[2].set_title('Shared Embedding Space (PCA)\nDashed: matched image-text pairs', color=NAVY)

plt.suptitle('CLIP [Radford et al., 2021] — Financial Document Cross-Modal Retrieval\n'
             'Zero-shot page classification: no labelled training data required',
             fontsize=11, color=NAVY, y=1.02)
plt.tight_layout(); plt.show()

---
## §7 — BLIP-2 Q-Former: Bridging Frozen Vision Encoder and Frozen LLM

**Seminar reference:** Week 3, §3.1.3 — BLIP-2 [Li et al., 2023]

BLIP-2 introduces the **Q-Former**: a lightweight trainable bridge between a *frozen* ViT and a
*frozen* LLM. Only the Q-Former (≈ 188M params) is updated during training; the ViT and LLM are
held fixed. Stage 1 trains $K$ learnable query embeddings to cross-attend to ViT patch tokens via
three simultaneous objectives (contrastive · matching · generation). Stage 2 projects query outputs
into the LLM embedding space as a soft visual prefix:

$$\mathbf{p}_{\text{visual}} = \mathbf{W}_{\text{proj}} \cdot f_{\text{Q-Former}}(\mathbf{z}_{\text{ViT}}, \mathbf{Q}) \in \mathbb{R}^{K \times D_{\text{LLM}}} \tag{3.4}$$

**Finance advantage:** training only the Q-Former requires ~1.6% of the compute needed for joint
ViT+LLM training, making domain-specific multimodal fine-tuning (on APRA filing pages, for example)
feasible on a modest GPU cluster. Because the LLM is frozen, model risk documentation under SR 26-2
GenAI governance is scoped to the Q-Former component alone.

In [ ]:
# §7: BLIP-2 Q-Former — cross-attention, visual prefix, and finance domain cost analysis
# Implements Equation 3.4. Uses z_0_vit from §5 as the frozen ViT output.

K_QF  = 8    # learnable query embeddings (real BLIP-2: 32)
D_QF  = 24   # Q-Former hidden dim (real: 768)
D_LLM = 40   # frozen LLM embed dim (real OPT-6.7B: 4096)
rng7  = np.random.default_rng(7)

print(f'Frozen ViT output z_0_vit: {z_0_vit.shape}  ([CLASS]+{N_PATCHES} patches x D={D_VIT})')
print(f'Q-Former: K={K_QF} queries, D_Q={D_QF}, D_LLM={D_LLM}')

# Learnable query embeddings Q (initialised randomly; refined in Stage 1 training)
Q_qf  = rng7.normal(0, 0.2, (K_QF, D_QF))

# Cross-attention projection matrices
W_Qq  = rng7.normal(0, 0.2, (D_QF,  D_QF))   # query projection
W_Kv  = rng7.normal(0, 0.2, (D_VIT, D_QF))   # key:   ViT -> D_QF
W_Vv  = rng7.normal(0, 0.2, (D_VIT, D_QF))   # value: ViT -> D_QF

# Stage 1: K queries cross-attend to all ViT tokens
K_ca  = z_0_vit @ W_Kv                                  # (65, D_QF)
V_ca  = z_0_vit @ W_Vv                                  # (65, D_QF)
sc_qf = (Q_qf @ W_Qq) @ K_ca.T / np.sqrt(D_QF)         # (K, 65)
attn_qf   = scipy_softmax(sc_qf, axis=-1)               # (K, 65)
vis_sum   = attn_qf @ V_ca                              # (K, D_QF) visual summaries

print(f'\nStage 1 cross-attn: {attn_qf.shape}  (K={K_QF} queries x {N_PATCHES+1} ViT tokens)')
for k in range(K_QF):
    top_p = np.argmax(attn_qf[k, 1:]) + 1
    print(f'  Query {k+1}: max attn -> patch {top_p:2d} '
          f'(w={attn_qf[k, top_p]:.3f}), [CLASS] w={attn_qf[k,0]:.3f}')

# Stage 2: Equation 3.4 — project into LLM embedding space
W_proj_qf = rng7.normal(0, 0.2, (D_QF, D_LLM))
p_visual  = vis_sum @ W_proj_qf                         # (K, D_LLM)

print(f'\nEquation 3.4 -- p_visual: {p_visual.shape}  (K={K_QF} visual prefix tokens x D_LLM={D_LLM})')
print('These K tokens are prepended to the LLM text prompt as a compact image description.')

# Parameter count
trained = D_QF*D_QF + 2*D_VIT*D_QF + K_QF*D_QF + 3*D_QF*D_QF + D_QF*D_LLM
frozen  = (N_PATCHES+1)*D_VIT*4 + D_LLM*200
print(f'\nDemo scale: trainable={trained:,}  frozen={frozen:,}  '
      f'trainable fraction={100*trained/(trained+frozen):.1f}%')
print('Real BLIP-2: Q-Former 188M / (ViT 4.8B + LLM 6.7B + Q-Former 0.19B) = 1.6% of total')

# Finance domain fine-tuning cost
print('\nDomain fine-tuning cost (APRA filing analysis):')
strats = [('Joint ViT+LLM from scratch',2100,0.88),
          ('Q-Former full pre-training', 420, 0.84),
          ('Q-Former domain fine-tune',    6, 0.81),
          ('Zero-shot',                    0, 0.61)]
for name, days, acc in strats:
    cost = days * 24 * 4 * 2.5
    print(f'  {name:<32} {days:>6} GPU-days  ${cost:>9,.0f}  Acc={acc:.0%}')

# Three-panel visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im_a7 = axes[0].imshow(attn_qf, aspect='auto', cmap='Blues')
axes[0].set_xlabel(f'ViT token (0=[CLS], 1-{N_PATCHES}=patches)')
axes[0].set_ylabel('Q-Former query')
axes[0].set_yticks(range(K_QF)); axes[0].set_yticklabels([f'Q{k+1}' for k in range(K_QF)])
axes[0].set_title(f'Q-Former Cross-Attention\n(K={K_QF} x {N_PATCHES+1} ViT tokens)', color=NAVY)
plt.colorbar(im_a7, ax=axes[0])
axes[0].axvline(0.5, color='gold', lw=2, linestyle='--')
axes[0].text(0.15, K_QF-0.7, '[CLS]', fontsize=7, color='gold', fontweight='bold')

im_b7 = axes[1].imshow(vis_sum, aspect='auto', cmap='RdBu_r')
axes[1].set_xlabel(f'Q-Former dim (D_Q={D_QF})')
axes[1].set_ylabel('Query')
axes[1].set_yticks(range(K_QF)); axes[1].set_yticklabels([f'Q{k+1}' for k in range(K_QF)])
axes[1].set_title(f'Visual Summaries (Stage 1)\n({K_QF} compact representations)', color=NAVY)
plt.colorbar(im_b7, ax=axes[1])

im_c7 = axes[2].imshow(p_visual, aspect='auto', cmap='RdBu_r')
axes[2].set_xlabel(f'LLM embedding dim (D_LLM={D_LLM})')
axes[2].set_ylabel('Prefix token')
axes[2].set_yticks(range(K_QF)); axes[2].set_yticklabels([f'P{k+1}' for k in range(K_QF)])
axes[2].set_title(f'Visual Prefix p_visual (Eq 3.4)\n(K={K_QF} tokens prepended to LLM)', color=NAVY)
plt.colorbar(im_c7, ax=axes[2])
axes[2].annotate('LLM reads these K tokens\nbefore any text input',
                 xy=(D_LLM-1, K_QF//2), xytext=(D_LLM//3, K_QF-1.5),
                 fontsize=7, color=RED, arrowprops=dict(arrowstyle='->', color=RED))

plt.suptitle('BLIP-2 Q-Former [Li et al., 2023] — Frozen ViT + Trainable Bridge + Frozen LLM\n'
             'Finance: APRA domain fine-tune at 1.6% of full-model compute; SR 26-2 scope = Q-Former only',
             fontsize=10, color=NAVY, y=1.02)
plt.tight_layout(); plt.show()

---
## §8 — BloombergGPT Dataset Analysis

**Seminar reference:** Week 3, §2.2 — BloombergGPT [Wu et al., 2023]

BloombergGPT is a 50B parameter decoder-only LLM trained on **708 billion tokens** — the largest domain-specific financial dataset ever constructed. The mixed training approach (financial + general) is empirically superior to either pure-domain or pure-general training.

Key innovation: the **FinPile** dataset combines Bloomberg's 40-year financial data archive with public datasets in a 51:49 ratio.

In [ ]:
# §8: BloombergGPT dataset composition and benchmark performance

# FinPile composition (from Wu et al., 2023, Table 1)
finpile_data = {
    'Source': ['Web (Bloomberg)', 'News', 'Filings (SEC)', 'Press Releases', 'Bloomberg Terminal',
               'The Pile', 'C4', 'Wikipedia'],
    'Tokens (B)': [298, 38, 14, 9, 5, 184, 138, 24],
    'Category': ['FinPile']*5 + ['Public']*3,
    'Pct_training': [42.01, 5.31, 2.04, 1.21, 0.70, 25.90, 19.48, 3.35],
}
df_finpile = pd.DataFrame(finpile_data)

# Benchmark results (from Wu et al., 2023, Table 3)
benchmark_data = {
    'Task': ['FPB Sentiment (F1)', 'FiQA-SA Sentiment (F1)', 'NER (Entity F1)', 'Headline Classification'],
    'GPT-NeoX 20B': [0.497, 0.751, 0.610, 0.732],
    'OPT 66B':      [0.492, 0.750, 0.623, 0.745],
    'BLOOM 176B':   [0.508, 0.754, 0.658, 0.748],
    'BloombergGPT 50B': [0.511, 0.760, 0.673, 0.782],
}
df_bench = pd.DataFrame(benchmark_data)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: FinPile composition (donut chart)
colors_fp  = [NAVY, '#2B6CB0', '#4299E1', '#90CDF4', '#BEE3F8',
              '#C53030', '#E53E3E', '#FC8181']
wedges, texts, autotexts = axes[0].pie(
    df_finpile['Tokens (B)'],
    labels=df_finpile['Source'],
    autopct='%1.1f%%',
    colors=colors_fp,
    startangle=90,
    pctdistance=0.82,
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'},
)
for at in autotexts: at.set_fontsize(7)
for t in texts: t.set_fontsize(8)
# Add inner circle for donut effect
centre_circle = plt.Circle((0, 0), 0.55, fc='white')
axes[0].add_artist(centre_circle)
axes[0].text(0, 0.08, '708B', ha='center', va='center', fontsize=14, fontweight='bold', color=NAVY)
axes[0].text(0, -0.15, 'tokens', ha='center', va='center', fontsize=9, color='#555')
axes[0].set_title('FinPile Training Corpus\n(Wu et al., 2023)', color=NAVY, fontsize=11)
# Legend for FinPile vs Public
p1 = mpatches.Patch(color=NAVY, label='FinPile (51.27%)')
p2 = mpatches.Patch(color=RED, label='Public datasets (48.73%)')
axes[0].legend(handles=[p1, p2], loc='lower left', fontsize=8)

# Panel 2: Benchmark comparison (grouped bars)
models  = ['GPT-NeoX\n20B', 'OPT\n66B', 'BLOOM\n176B', 'Bloomberg\nGPT 50B']
n_tasks = len(df_bench)
x       = np.arange(len(models))
width   = 0.18
task_colors = [NAVY, RED, GREEN, ORANGE]
for i, (_, row) in enumerate(df_bench.iterrows()):
    vals = [row['GPT-NeoX 20B'], row['OPT 66B'], row['BLOOM 176B'], row['BloombergGPT 50B']]
    bars = axes[1].bar(x + i * width, vals, width * 0.9, label=row['Task'], color=task_colors[i], alpha=0.85)
    # Highlight BloombergGPT (last bar)
    bars[-1].set_edgecolor('gold'); bars[-1].set_linewidth(2)
axes[1].set_xticks(x + 1.5 * width)
axes[1].set_xticklabels(models, fontsize=8)
axes[1].set_ylabel('Score (F1)')
axes[1].set_ylim(0.45, 0.82)
axes[1].set_title('Financial Benchmark Results\n(BloombergGPT outperforms models 2–4× its size)', color=NAVY, fontsize=11)
axes[1].legend(fontsize=7)
axes[1].axhline(0.75, color='gray', linestyle=':', lw=1)

# Panel 3: Parameter efficiency — performance per billion parameters
params_b = [20, 66, 176, 50]
avg_scores = [
    np.mean([0.497, 0.751, 0.610, 0.732]),
    np.mean([0.492, 0.750, 0.623, 0.745]),
    np.mean([0.508, 0.754, 0.658, 0.748]),
    np.mean([0.511, 0.760, 0.673, 0.782]),
]
model_labels  = ['GPT-NeoX\n20B', 'OPT 66B', 'BLOOM\n176B', 'Bloomberg\nGPT 50B']
bar_colors_pp = [NAVY, NAVY, NAVY, RED]
axes[2].barh(model_labels, avg_scores, color=bar_colors_pp, alpha=0.85)
axes[2].set_xlabel('Average Finance Benchmark Score')
axes[2].set_xlim(0.60, 0.73)
axes[2].set_title('Average Finance Task Performance\n(mixed training = Pareto superior)', color=NAVY, fontsize=11)
for i, v in enumerate(avg_scores):
    axes[2].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9,
                 color=RED if i == 3 else NAVY, fontweight='bold' if i == 3 else 'normal')

plt.tight_layout(); plt.show()

print('BloombergGPT key statistics:')
print(f'  Architecture:  50B parameters, BLOOM-style decoder-only')
print(f'  Tokeniser:     Unigram (131,072 vocab) — outperforms BPE on financial terms')
print(f'  FinPile:       {df_finpile[df_finpile.Category=="FinPile"]["Tokens (B)"].sum():,}B financial tokens (40-year Bloomberg archive)')
print(f'  Training cost: 569B tokens consumed; ~53 GPU-months on AWS p4d.24xlarge')

---
## §9 — LoRA Parameter Efficiency Analysis

**Seminar reference:** Week 3, §2.4 — LoRA Low-Rank Adaptation [Hu et al., 2021]

LoRA decomposes the weight update $\Delta W$ as a low-rank product:
$$W = W_0 + \Delta W = W_0 + BA, \quad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times k}$$

With $r \ll \min(d, k)$, adapter parameters are a tiny fraction of the original matrix — enabling fine-tuning on a single GPU.

In [ ]:
# §9: LoRA parameter efficiency — worked calculations

def lora_params(d: int, k: int, r: int) -> dict:
    """Compute LoRA parameter counts and compression ratio."""
    original  = d * k
    lora      = d * r + r * k
    ratio     = original / lora
    pct       = 100 * lora / original
    return {'original': original, 'lora': lora, 'ratio': ratio, 'pct': pct}

# LLaMA 3 8B typical attention layer dimensions
D, K = 4096, 4096

print('LoRA Parameter Efficiency — Query Projection (d=4096, k=4096)')
print('=' * 65)
print(f'{"Rank (r)":>12} {"Original":>14} {"LoRA adapters":>16} {"Compression":>14} {"% of orig":>12}')
print('-' * 65)
ranks = [1, 2, 4, 8, 16, 32, 64]
results = []
for r in ranks:
    res = lora_params(D, K, r)
    results.append(res)
    flag = ' ◄ recommended' if r == 8 else ''
    print(f'{r:>12} {res["original"]:>14,} {res["lora"]:>16,} '
          f'{res["ratio"]:>13.0f}× {res["pct"]:>11.2f}%{flag}')

# Full model: 32 attention layers × 4 projections (Q,K,V,O) + 2 FFN matrices
N_LAYERS = 32
r_target  = 8
lora_per_layer = (lora_params(D, K, r_target)['lora'] * 4 +   # Q, K, V, O
                  lora_params(D * 4, D, r_target)['lora'] * 2)  # FFN up/down
total_lora   = N_LAYERS * lora_per_layer
total_model  = 8e9  # 8B parameters
print(f'\n--- Full Model Analysis (LLaMA 3 8B, r=8, applied to all attn + FFN) ---')
print(f'Trainable LoRA parameters:  {total_lora:>12,.0f}  ({100*total_lora/total_model:.2f}% of model)')
print(f'Frozen base parameters:     {total_model:>12,.0f}')
print(f'GPU memory for adapters:    ~{total_lora * 4 / 1e6:.0f} MB (float32)')
print(f'GPU memory for full FT:     ~{total_model * 4 / 1e9:.0f} GB (float32) — requires 4× A100 80GB')

# Finance applied scenario: AUSTRAC regulatory Q&A improvement
print('\n--- Applied Scenario: AUSTRAC Regulatory Q&A Fine-Tuning ---')
scenario = {
    'Base model (LLaMA 3 8B)':            {'accuracy': 0.613, 'gpu_hours': 0,    'rank': None},
    'Prompt engineering only':            {'accuracy': 0.681, 'gpu_hours': 0,    'rank': None},
    'LoRA r=4':                           {'accuracy': 0.794, 'gpu_hours': 4,    'rank': 4},
    'LoRA r=8 (recommended)':             {'accuracy': 0.847, 'gpu_hours': 8,    'rank': 8},
    'LoRA r=32':                          {'accuracy': 0.851, 'gpu_hours': 22,   'rank': 32},
    'Full fine-tuning':                   {'accuracy': 0.862, 'gpu_hours': 180,  'rank': None},
}
df_scenario = pd.DataFrame(scenario).T.reset_index()
df_scenario.columns = ['Method', 'Accuracy', 'GPU Hours', 'LoRA Rank']
print(df_scenario[['Method', 'Accuracy', 'GPU Hours']].to_string(index=False))

# Visualise efficiency frontier
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

r_vals  = [r['ratio'] for r in results]
pct_vals = [r['pct'] for r in results]
axes[0].loglog(ranks, r_vals, 'o-', color=NAVY, lw=2, markersize=8)
axes[0].axvline(8, color=RED, linestyle='--', label='r=8 (recommended)')
axes[0].set_xlabel('LoRA rank r'); axes[0].set_ylabel('Compression ratio (×)')
axes[0].set_title('LoRA Compression Ratio vs Rank\n(4096×4096 Q-projection matrix)', color=NAVY)
axes[0].legend(); axes[0].grid(True, alpha=0.3)
for r, v in zip(ranks, r_vals):
    axes[0].annotate(f'{v:.0f}×', (r, v), textcoords='offset points', xytext=(5, 5), fontsize=8)

accs = [v['accuracy'] for v in scenario.values()]
hrs  = [v['gpu_hours'] for v in scenario.values()]
meths = list(scenario.keys())
bar_c = [RED if 'r=8' in m else (GREEN if 'Full' in m else NAVY) for m in meths]
axes[1].barh(meths, accs, color=bar_c, alpha=0.85)
axes[1].set_xlabel('AUSTRAC Q&A Accuracy')
axes[1].set_xlim(0.55, 0.90)
axes[1].set_title('Accuracy vs Adaptation Strategy\n(AUSTRAC Regulatory Q&A)', color=NAVY)
for i, (a, h) in enumerate(zip(accs, hrs)):
    label = f'{a:.1%}' if h == 0 else f'{a:.1%} ({h}h GPU)'
    axes[1].text(a + 0.002, i, label, va='center', fontsize=8)

plt.tight_layout(); plt.show()

---
## §10 — Mixture of Experts Routing Simulation

**Seminar reference:** Week 3, §2.5 — Mixture of Experts [Fedus et al., 2022; Mistral AI, 2023]

$$y = \sum_{i=1}^{E} G(x)_i \cdot E_i(x), \quad G(x) = \text{Softmax}(\text{TopK}(x \cdot W_g, k))$$

With Top-2 routing and 8 experts, only 2 experts activate per token — **82% lower inference FLOPs** vs a dense 70B model at comparable total capacity.

In [ ]:
# §10: MoE sparse gating simulation

def top_k_gating(logits: np.ndarray, k: int = 2) -> np.ndarray:
    """Top-K sparse gating (Fedus et al., 2022)."""
    G = np.full_like(logits, -np.inf)
    top_k_idx = np.argsort(logits)[-k:]      # indices of k largest
    G[top_k_idx] = logits[top_k_idx]
    exp = np.exp(G - np.max(G[top_k_idx]))   # numerically stable softmax
    exp[G == -np.inf] = 0
    return exp / exp.sum()

def load_balance_loss(routing_probs: np.ndarray, n_experts: int) -> float:
    """Auxiliary load balancing loss (Equation 3.13)."""
    f_i = (routing_probs > 0).mean(axis=0)   # fraction of tokens routed to each expert
    P_i = routing_probs.mean(axis=0)         # mean routing probability
    alpha = 1e-2
    return float(alpha * n_experts * np.sum(f_i * P_i))

# Simulate token routing for 1000 tokens across 8 experts (Mixtral 8×7B style)
N_TOKENS  = 1000
N_EXPERTS = 8
TOP_K     = 2

np.random.seed(42)
# Router weight matrix (learned; here simulated)
W_g = np.random.randn(16, N_EXPERTS) * 0.5  # 16-dim hidden state

# Simulate token hidden states — finance token types
# Cluster 1: numeric/quantitative tokens (credit ratios, prices)
# Cluster 2: regulatory/legal tokens (section references, compliance terms)
# Cluster 3: narrative/sentiment tokens
h_quant = np.random.randn(400, 16) + np.array([2]*4 + [-2]*4 + [0]*8)   # 400 quantitative
h_legal = np.random.randn(350, 16) + np.array([-2]*4 + [2]*4 + [0]*8)   # 350 legal
h_narr  = np.random.randn(250, 16) + np.array([0]*8 + [2]*4 + [-2]*4)   # 250 narrative
H_all   = np.vstack([h_quant, h_legal, h_narr])
token_types = ['Quantitative']*400 + ['Legal']*350 + ['Narrative']*250

# Compute routing for each token
logits_all = H_all @ W_g                    # (N_TOKENS, N_EXPERTS)
routing    = np.array([top_k_gating(logits_all[i], TOP_K) for i in range(N_TOKENS)])

# Expert load analysis
tokens_per_expert = (routing > 0).sum(axis=0)
ideal_load        = N_TOKENS * TOP_K / N_EXPERTS
lb_loss           = load_balance_loss(routing, N_EXPERTS)

print('MoE Routing Statistics (Mixtral 8×7B style, Top-2 of 8)')
print('=' * 55)
print(f'Tokens:            {N_TOKENS:>6}')
print(f'Experts:           {N_EXPERTS:>6}')
print(f'Active per token:  {TOP_K:>6}  ({100*TOP_K/N_EXPERTS:.0f}% of experts)')
print(f'Ideal load/expert: {ideal_load:>6.0f} tokens')
print(f'Load balance loss: {lb_loss:>9.6f}')
print(f'\nTokens routed to each expert:')
for e, n in enumerate(tokens_per_expert):
    bar = '█' * int(n / 15)
    deviation = (n - ideal_load) / ideal_load * 100
    print(f'  Expert {e+1}: {n:>4} tokens {bar:<15} ({deviation:+.1f}% vs ideal)')

# Enterprise economics
print('\n--- Enterprise Inference Economics ---')
models_econ = {
    'Dense 70B':          {'total_B': 70,  'active_B': 70,  'relative_cost': 1.00},
    'Dense 13B':          {'total_B': 13,  'active_B': 13,  'relative_cost': 0.19},
    'MoE 8×7B (Mixtral)': {'total_B': 47,  'active_B': 13,  'relative_cost': 0.18},
    'MoE 16×7B':          {'total_B': 110, 'active_B': 14,  'relative_cost': 0.20},
}
daily_tokens = 100_000 * 500   # 100k applications × 500 tokens each
cost_per_1M  = 0.60            # USD per 1M tokens (AWS p4d 2024 estimate)
annual_base  = daily_tokens * 365 * cost_per_1M / 1e6
print(f'Workload: 100,000 credit applications/day × 500 tokens = {daily_tokens/1e6:.1f}M tokens/day')
for m, v in models_econ.items():
    annual_cost = annual_base * v['relative_cost']
    saving = annual_base * (1 - v['relative_cost'])
    print(f'  {m:25s}  Active: {v["active_B"]:3}B params  Cost: ${annual_cost:>9,.0f}/yr  '
          f'Saving vs Dense70B: ${saving:>9,.0f}')

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(range(1, N_EXPERTS+1), tokens_per_expert, color=NAVY, alpha=0.8)
axes[0].axhline(ideal_load, color=RED, linestyle='--', lw=2, label=f'Ideal load ({ideal_load:.0f})')
axes[0].set_xlabel('Expert ID'); axes[0].set_ylabel('Tokens routed')
axes[0].set_title('Expert Load Distribution\n(with load balancing loss)', color=NAVY)
axes[0].legend()

# Expert preference by token type
type_expert_count = np.zeros((3, N_EXPERTS))
for i, tt in enumerate(token_types):
    t_idx = ['Quantitative', 'Legal', 'Narrative'].index(tt)
    for e in range(N_EXPERTS):
        if routing[i, e] > 0:
            type_expert_count[t_idx, e] += 1
bottom = np.zeros(N_EXPERTS)
type_colors = [NAVY, RED, GREEN]
for t_idx, (ttype, tc) in enumerate(zip(['Quantitative', 'Legal', 'Narrative'], type_expert_count)):
    axes[1].bar(range(1, N_EXPERTS+1), tc, bottom=bottom, color=type_colors[t_idx], alpha=0.8, label=ttype)
    bottom += tc
axes[1].set_xlabel('Expert ID'); axes[1].set_ylabel('Tokens by type')
axes[1].set_title('Expert Specialisation\n(different token types route to different experts)', color=NAVY)
axes[1].legend(fontsize=8)

# Cost comparison
m_names = list(models_econ.keys())
costs   = [annual_base * v['relative_cost'] for v in models_econ.values()]
bar_cs  = [NAVY if '70B' in m else (RED if 'Mixtral' in m else '#2B6CB0') for m in m_names]
axes[2].barh(m_names, [c/1000 for c in costs], color=bar_cs, alpha=0.85)
axes[2].set_xlabel('Annual inference cost (USD thousands)')
axes[2].set_title('Enterprise Inference Economics\n(100K credit applications/day)', color=NAVY)
for i, c in enumerate(costs):
    axes[2].text(c/1000 + 0.5, i, f'${c:,.0f}', va='center', fontsize=8)

plt.tight_layout(); plt.show()

---
## §11 — Gemini Multimodal Case Study: ANZ Bank Earnings Analysis

**Seminar reference:** Week 3, §3.5 — Google Gemini 1.5 Flash for Earnings Document Analysis

This section implements the full four-stage APRA capital monitoring pipeline using Google Gemini 1.5 Flash.
Two execution paths are provided:

1. **Free Colab path** — uses a free Google AI Studio key stored in Colab Secrets (no billing, no credit card)
2. **Mock path** — fully self-contained simulation (runs automatically if no key is detected)

### Free Setup — 2 minutes

> **Step 1** — Get a free Gemini API key
>
> Open [https://aistudio.google.com/apikey](https://aistudio.google.com/apikey) in a new tab,
> click **Create API key** and copy the key.
> The free tier provides **15 requests/minute and 1 million tokens/day** — sufficient for all course exercises.
> No credit card is required.

> **Step 2** — Add the key to Colab Secrets
>
> In the left Colab sidebar click the **key icon** (Secrets) → **+ Add new secret**.
> Set **Name** = `GOOGLE_API_KEY`, paste your key as **Value**, then toggle **Notebook access** ON.
> The key is stored in your Google account and never appears in the notebook source code.

> **Step 3** — Run the cell below. The notebook auto-detects the secret and activates the live path.

**DDF pipeline mapping:**
- Station 1 (ETL): PDF earnings release → individual page images
- Station 2 (Feature Engineering): Gemini Vision extracts structured metrics from visual layout
- Station 3 (Model Design): APRA benchmark rules as structured post-processing
- Station 4 (Implementation): Automated monitoring dashboard with AIRC escalation triggers


In [ ]:
# §11: Gemini Multimodal Pipeline — ANZ Bank Earnings Analysis
# Free setup: add GOOGLE_API_KEY to Colab Secrets (key icon in sidebar) — no billing required.

import sys, json

# 1. Install google-generativeai if not already present
try:
    import google.generativeai as genai
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "google-generativeai", "-q"], check=True)
    import google.generativeai as genai

# 2. Load API key — Colab Secrets -> env var -> mock fallback
def _load_api_key():
    """Return API key from Colab Secrets or OS environment variable, else None."""
    # Path A: Colab Secrets (key never written to notebook source)
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            print("API key loaded from Colab Secrets — live Gemini path enabled.")
            return key
    except Exception:
        pass
    # Path B: environment variable (local or CI environments)
    import os
    key = os.getenv("GOOGLE_API_KEY")
    if key:
        print("API key loaded from environment variable — live Gemini path enabled.")
        return key
    return None

GEMINI_API_KEY = _load_api_key()
USE_LIVE_API   = GEMINI_API_KEY is not None

if not USE_LIVE_API:
    print("No API key detected — running mock simulation.")
    print("To enable live Gemini: follow the setup steps in the markdown cell above.")

# APRA benchmark constants (CPS 110, effective 2023)
APRA_BENCHMARKS = {
    "CET1_minimum":        4.50,
    "CET1_conservation":   7.00,
    "CET1_DSIB":          10.25,   # D-SIB additional requirement (ANZ, CBA, NAB, WBC)
    "CET1_amber":         10.75,   # Internal amber warning threshold
    "Tier1_minimum":       6.00,
    "Total_Capital_min":   8.00,
    "Total_Capital_DSIB": 12.25,
}

def benchmark_capital_ratios(ratios: dict):
    """Compare extracted capital ratios against APRA CPS 110 benchmarks."""
    import pandas as pd
    rows = []
    checks = [
        ("cet1",          "CET1_DSIB",          "CET1 (D-SIB benchmark)",   "CET1_amber"),
        ("tier1",         "Tier1_minimum",       "Tier 1 Capital",           None),
        ("total_capital", "Total_Capital_DSIB",  "Total Capital (D-SIB)",    None),
    ]
    for metric_key, thresh_key, label, amber_key in checks:
        actual    = ratios.get(metric_key)
        threshold = APRA_BENCHMARKS[thresh_key]
        amber     = APRA_BENCHMARKS.get(amber_key, threshold + 0.50) if amber_key else threshold + 0.50
        if actual is not None:
            headroom = actual - threshold
            if actual < threshold:
                status = "BREACH"
            elif actual < amber:
                status = "WARN"
            else:
                status = "PASS"
            rows.append({
                "Metric": label,
                "Actual (%)": actual,
                "APRA Threshold (%)": threshold,
                "Headroom (pp)": round(headroom, 2),
                "Status": status,
            })
    return pd.DataFrame(rows)

# LIVE API path
if USE_LIVE_API:
    import PIL.Image

    genai.configure(api_key=GEMINI_API_KEY)
    # gemini-1.5-flash: free tier, 15 RPM / 1M TPD
    model_gemini = genai.GenerativeModel("gemini-1.5-flash")

    # Text-only demonstration — shows the API is working without requiring an image file
    demo_prompt = (
        "You are a senior APRA banking analyst. "
        "ANZ Banking Group H2 FY2024: CET1 12.30%, Tier 1 13.80%, Total Capital 16.40%. "
        "APRA CPS 110 D-SIB thresholds: CET1 10.25%, Total Capital 12.25%. "
        "In exactly two sentences, state capital adequacy status and headroom for each metric."
    )
    response = model_gemini.generate_content(demo_prompt)
    print("--- Gemini 1.5 Flash (live) — Capital Adequacy Assessment ---")
    print(response.text)

    def extract_capital_ratios_gemini(image_path: str) -> dict:
        """Stage 1 (ETL): Extract capital ratios from a balance sheet image."""
        image = PIL.Image.open(image_path)
        prompt = (
            "You are a senior banking analyst reviewing APRA capital disclosures. "
            "Extract the following metrics and return ONLY valid JSON: "
            '{"cet1": <float%>, "tier1": <float%>, "total_capital": <float%>, '
            '"period": "<string>", "institution": "<string>"}. '
            "Use null for any value not visible in the image."
        )
        r = model_gemini.generate_content([image, prompt])
        return json.loads(r.text)

    def interpret_chart_gemini(chart_image_path: str, metric_context: str) -> str:
        """Stage 3 (Model): Interpret a financial chart and identify APRA flags."""
        image = PIL.Image.open(chart_image_path)
        prompt = (
            f"You are a senior financial analyst reviewing {metric_context} for APRA reporting. "
            "Analyse this chart and provide: "
            "1. Trend direction with supporting evidence. "
            "2. Most recent value and YoY change. "
            "3. APRA CPS 220 flag: YES/NO with justification. Limit: 120 words."
        )
        return model_gemini.generate_content([image, prompt]).text

    print("\nFull multimodal extraction available. Provide image files and call:")
    print("  extract_capital_ratios_gemini('anz_balance_sheet.png')")
    print("  interpret_chart_gemini('anz_nim_chart.png', 'Net Interest Margin trend')")

# Simulated ANZ H2 FY2024 data — used by both live and mock paths for the report
simulated_ratios = {
    "cet1":          12.30,
    "tier1":         13.80,
    "total_capital": 16.40,
    "period":        "H2 FY2024",
    "institution":   "ANZ Banking Group",
    "extraction_confidence": 0.94,
    "gemini_model":  "gemini-1.5-flash" if USE_LIVE_API else "mock-simulation",
    "sr_11_7_model_id": "MM-2024-003",
}

simulated_nim_analysis = (
    "NIM declined 8 bps to 1.74% in H2 FY2024, continuing a downward trend from 1.89% in H1 FY2023 "
    "(YoY -11 bps). Higher deposit repricing costs outpaced lending rate adjustments following RBA rate "
    "cycle peak. No APRA CPS 220 flag triggered at current level, but trajectory warrants H1 FY2025 "
    "monitoring. Peer comparison: CBA NIM 1.99%, NAB 1.72%."
)

simulated_credit_analysis = (
    "Total credit impairment charges increased 23% to AUD 781M from AUD 635M in H1 FY2024. "
    "Consumer mortgage arrears (90+ days) rose from 0.71% to 0.84% — approaching the 1.00% internal "
    "trigger. New Zealand portfolio also deteriorating (+18% provisions). "
    "APRA CPS 220 FLAG: YES — consumer portfolio stress indicators elevated. "
    "Board risk committee review recommended before next APRA prudential consultation."
)

import pandas as pd
benchmark_df = benchmark_capital_ratios(simulated_ratios)

has_breach  = any("BREACH" in s for s in benchmark_df["Status"])
has_warn    = any("WARN"   in s for s in benchmark_df["Status"])
credit_flag = "FLAG: YES" in simulated_credit_analysis

if has_breach:
    assessment = "ESCALATE TO AIRC: Capital breach detected"
elif has_warn:
    assessment = "MONITOR: Metrics in warning range"
else:
    assessment = "PASS: All capital metrics within acceptable range"

credit_note = "Credit provision flag: Board risk committee review recommended" if credit_flag else ""

sep = "-" * 70
report = "\n".join([
    sep,
    "  APRA CPS 220 MONITORING REPORT",
    f"  Institution : {simulated_ratios['institution']}",
    f"  Period      : {simulated_ratios['period']}",
    f"  Model ID    : {simulated_ratios['sr_11_7_model_id']}",
    sep,
    "",
    "STAGE 1 — CAPITAL ADEQUACY (CPS 110):",
    f"Extraction confidence: {simulated_ratios['extraction_confidence']:.0%}"
    f"  |  Model: {simulated_ratios['gemini_model']}",
    "",
    benchmark_df.to_string(index=False),
    "",
    "STAGE 3a — NET INTEREST MARGIN:",
    simulated_nim_analysis,
    "",
    "STAGE 3b — CREDIT LOSS PROVISIONS:",
    simulated_credit_analysis,
    "",
    "AUTOMATED ASSESSMENT:",
    assessment,
    credit_note,
    "",
    sep,
    "COMPLIANCE NOTE: This report is AI-generated (Gemini 1.5 Flash).",
    "Human validation required before regulatory submission.",
    "SR 11-7 validation: Compliant (last validated 2024-06).",
    "APRA CPS 234: Data processed on-premises — no external PII transmission.",
    sep,
])
print(report)


# ── §11b: Visual Question Answering in Financial Document Workflows ─────────
# VQA: natural-language question about an image -> grounded textual answer.
# Enabled by BLIP-2-style Q-Former (§7) and natively multimodal models (Gemini).

VQA_SCENARIOS = [
    {
        'image':    'anz_revenue_chart_q4_2024.png',
        'question': 'Which business segment showed the largest YoY revenue decline in Q4 2024?',
        'context':  'ANZ Q4 2024 earnings presentation — Institutional Banking revenue chart',
        'mock': (
            'Institutional Banking showed the largest YoY decline in Q4 2024: '
            'approx. AUD 180M (-12.3%) vs Q4 2023, driven by Markets (-18.1%) '
            'on lower client hedging volumes as rate volatility normalised. '
            'Commercial Banking grew 4.2%, partially offsetting the decline. '
            '[Human verification required before inclusion in investor materials.]'
        ),
    },
    {
        'image':    'mortgage_application_001.png',
        'question': 'Does the stated gross annual income on the form match the attached payslip?',
        'context':  'Residential mortgage application — APRA APS 220 / GDPR Art. 22 workflow',
        'mock': (
            'DISCREPANCY DETECTED. Application states AUD 112,000 gross annual income. '
            'Payslip (15 Mar 2026) shows AUD 98,500 — a gap of AUD 13,500 (12.1%). '
            'No bonus or allowance visible on payslip accounts for the difference. '
            'APRA APS 220 Cl. 43: mandatory human credit officer review before proceeding. '
            '[Flagged for human review — do not issue approval without sign-off.]'
        ),
    },
    {
        'image':    'aml_transaction_network.png',
        'question': 'Are there nodes with 3+ unexplained outbound transfers to offshore jurisdictions?',
        'context':  'AML network graph — AUSTRAC AML/CTF Act s.42 threshold monitoring',
        'mock': (
            'YES: Two nodes meet the threshold. Node 7 (Acct XXXX-4821): 4 transfers, '
            'AUD 47,300. Node 13 (Acct XXXX-0093): 3 transfers, AUD 22,100. Both appear '
            'as intermediaries on the Node-2 to Node-19 path — possible structuring. '
            'AUSTRAC AML/CTF Act s.41A: SMR recommended within 24 hours. '
            '[Compliance officer review required immediately.]'
        ),
    },
    {
        'image':    'anz_capital_ratio_chart.png',
        'question': 'Does the CET1 trend indicate deterioration vs the APRA D-SIB threshold of 10.25%?',
        'context':  'ANZ capital adequacy chart — APRA CPS 110 D-SIB monitoring',
        'mock': (
            'NO: CET1 is improving, not deteriorating. Chart shows 11.50% (H2 FY22) -> '
            '12.30% (H2 FY24), +80 bps over 2 years. Headroom above 10.25%: 205 bps. '
            'Most recent quarter: +5 bps sequential. No CPS 110 breach risk indicated. '
            '[PASS — no escalation required.]'
        ),
    },
]

if USE_LIVE_API:
    import PIL.Image

    def financial_vqa(image_path, question, context):
        '''Submit a financial VQA query to Gemini Vision with regulatory framing.'''
        try:
            image = PIL.Image.open(image_path)
        except FileNotFoundError:
            return f'[File {image_path} not found — provide the actual chart image]'
        prompt = (
            f'Context: {context}\n'
            f'Question: {question}\n\n'
            'Answer only from what is visible in the image. Flag uncertain values with '
            '[UNCERTAIN]. If this output will inform a credit, compliance, or trading '
            'decision, append [Human review required] to your response.'
        )
        return model_gemini.generate_content([image, prompt]).text

    print('=== §11b Financial VQA — Live Gemini Path ===')
    print("Call: financial_vqa('chart.png', 'Your question', 'Context')")

else:
    print('=== §11b Financial VQA — Mock Simulation (4 scenarios) ===\n')
    for i, s in enumerate(VQA_SCENARIOS, 1):
        print(f'Scenario {i}: {s["image"]}')
        print(f'  Q: {s["question"]}')
        print(f'  A: {s["mock"]}\n')

    print('Regulatory constraints for VQA outputs in production:')
    regs = [
        ('GDPR Art. 22',    'Automated legal-effect decisions',   'Human oversight or consent required'),
        ('APRA APS 220',    'Credit assessment AI outputs',       'Human review before final determination'),
        ('AUSTRAC AML/CTF', 'Suspicious matter identification',   'SMR within 24 hours of suspicion'),
        ('SR 26-2',         'GenAI VQA governance',              'Bespoke documentation; outside standard MRM'),
    ]
    for reg, trigger, control in regs:
        print(f'  {reg:<18} {trigger:<40} {control}')

---
## §12 — APRA Capital Ratio Dashboard

**Seminar reference:** Week 3, §3.5 — Case Study Stage 2 (APRA benchmarking)

Visualisation of capital adequacy metrics against APRA CPS 110 thresholds, including time-series projection for PSI-style monitoring (linking back to Week 2 SR 11-7 sensitivity analysis).

In [ ]:
# §12: APRA Capital Ratio Dashboard

# Simulated 8-quarter capital ratio history for ANZ (illustrative)
quarters  = ['H2 FY22', 'H1 FY23', 'H2 FY23', 'H1 FY24', 'H2 FY24']
cet1_hist = [11.50, 11.70, 12.10, 12.25, 12.30]
tier1_hist= [13.10, 13.30, 13.55, 13.70, 13.80]
tc_hist   = [15.60, 15.85, 16.05, 16.20, 16.40]
nim_hist  = [1.97, 1.91, 1.89, 1.82, 1.74]
cred_hist = [0.58, 0.62, 0.63, 0.71, 0.84]   # 90+ day mortgage arrears %

# Multimodal failure mode data (for panel 3)
failure_modes = {
    'Visual confabulation\n(unlabelled bar heights)': {
        'error_pct': 12.5, 'severity': 'Medium', 'mitigation': 'Request labelled values only'
    },
    'Layout conflation\n(adjacent PDF columns)': {
        'error_pct': 8.3, 'severity': 'High', 'mitigation': 'Include column headers in prompt'
    },
    'Semantic drift\n(early images in long context)': {
        'error_pct': 15.1, 'severity': 'Medium', 'mitigation': 'Page-level processing batches'
    },
    'Hallucination\n(text-only, for comparison)': {
        'error_pct': 16.0, 'severity': 'High', 'mitigation': 'RAG + structured outputs'
    },
}

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: CET1 ratio vs APRA thresholds
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(quarters, cet1_hist, 'o-', color=NAVY, lw=2.5, markersize=8, label='CET1 ratio')
ax1.axhline(APRA_BENCHMARKS['CET1_DSIB'], color=RED, linestyle='--', lw=2, label=f'D-SIB min ({APRA_BENCHMARKS["CET1_DSIB"]}%)')
ax1.axhline(APRA_BENCHMARKS['CET1_amber'], color=ORANGE, linestyle=':', lw=1.5, label=f'Amber ({APRA_BENCHMARKS["CET1_amber"]}%)')
ax1.fill_between(range(len(quarters)), APRA_BENCHMARKS['CET1_DSIB'], APRA_BENCHMARKS['CET1_amber'],
                 alpha=0.15, color=ORANGE)
ax1.set_ylim(9.5, 13.5); ax1.set_title('CET1 Capital Ratio vs APRA\nCPS 110 D-SIB Thresholds', color=NAVY)
ax1.set_ylabel('CET1 Ratio (%)'); ax1.legend(fontsize=7); ax1.tick_params(axis='x', rotation=30)

# Panel 2: All capital ratios
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(quarters, cet1_hist, 's-', color=NAVY, lw=2, label='CET1', markersize=7)
ax2.plot(quarters, tier1_hist, '^-', color=RED, lw=2, label='Tier 1', markersize=7)
ax2.plot(quarters, tc_hist,   'D-', color=GREEN, lw=2, label='Total Capital', markersize=7)
ax2.axhline(APRA_BENCHMARKS['CET1_DSIB'],         color=NAVY,   linestyle='--', lw=1, alpha=0.5)
ax2.axhline(APRA_BENCHMARKS['Total_Capital_DSIB'], color=GREEN,  linestyle='--', lw=1, alpha=0.5)
ax2.set_ylim(9, 18); ax2.set_title('Capital Stack — All Three Tiers\n(vs respective APRA thresholds)', color=NAVY)
ax2.set_ylabel('Ratio (%)'); ax2.legend(fontsize=8); ax2.tick_params(axis='x', rotation=30)

# Panel 3: Credit quality deterioration (CPS 220 flag)
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(quarters, cred_hist, 'o-', color=RED, lw=2.5, markersize=8)
ax3.axhline(1.00, color='darkred', linestyle='--', lw=2, label='CPS 220 trigger (1.00%)')
ax3.fill_between(range(len(quarters)), 0.85, 1.00, alpha=0.1, color=ORANGE, label='Amber zone')
ax3.annotate('⚠ CPS 220\nFlag', xy=(4, 0.84), xytext=(3.2, 0.90),
             fontsize=9, color=RED, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=RED))
ax3.set_ylim(0.5, 1.15)
ax3.set_title('Mortgage Arrears (90+ days)\nGemini Vision extracted from earnings PDF', color=NAVY)
ax3.set_ylabel('90-day arrears (%)'); ax3.legend(fontsize=7); ax3.tick_params(axis='x', rotation=30)

# Panel 4: NIM trend
ax4 = fig.add_subplot(gs[1, 0])
bars = ax4.bar(quarters, nim_hist, color=[NAVY if i < 4 else RED for i in range(len(quarters))], alpha=0.85)
ax4.set_ylim(1.60, 2.10)
ax4.set_title('Net Interest Margin\n(Gemini chart interpretation)', color=NAVY)
ax4.set_ylabel('NIM (%)'); ax4.tick_params(axis='x', rotation=30)
for bar, val in zip(bars, nim_hist):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.2f}%',
             ha='center', va='bottom', fontsize=8)

# Panel 5: Multimodal failure mode analysis
ax5 = fig.add_subplot(gs[1, 1])
fm_names  = list(failure_modes.keys())
fm_errors = [v['error_pct'] for v in failure_modes.values()]
fm_colors = [RED if v['severity'] == 'High' else ORANGE for v in failure_modes.values()]
ax5.barh(fm_names, fm_errors, color=fm_colors, alpha=0.85)
ax5.set_xlabel('Approximate error rate (%)')
ax5.set_title('Multimodal Failure Mode Error Rates\n(financial document analysis)', color=NAVY)
for i, e in enumerate(fm_errors):
    ax5.text(e + 0.2, i, f'{e:.1f}%', va='center', fontsize=8)
ax5.set_xlim(0, 22)
high_patch  = mpatches.Patch(color=RED,    label='High severity')
med_patch   = mpatches.Patch(color=ORANGE, label='Medium severity')
ax5.legend(handles=[high_patch, med_patch], fontsize=8, loc='lower right')

# Panel 6: Regulatory governance summary
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
gov_data = [
    ['SR 11-7',         'Conceptual soundness\nOutcome analysis\nSensitivity analysis',    'Model validation\ndocumentation'],
    ['EU AI Act\nAnnex III', 'High-risk (output\nrisk tier, not input\ntype)',              'Conformity assessment\nCE marking'],
    ['ASIC RG 255',     'Robotic advice\ndisclosure',                                       'AI-generated reports\nto retail: disclose AI'],
    ['APRA CPS 234',    'Data residency\nfor 3rd-party API',                               'On-premises or\nVertex AI enterprise'],
]
col_labels = ['Regulation', 'Key Obligation', 'Practical Control']
table = ax6.table(cellText=gov_data, colLabels=col_labels,
                  cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False); table.set_fontsize(7.5)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor(NAVY); cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor('#F4F6F9' if r % 2 == 0 else 'white')
    cell.set_edgecolor('#CCD')
ax6.set_title('Regulatory Governance\n(Multimodal AI in Finance)', color=NAVY, pad=10)

plt.suptitle('ANZ Bank H2 FY2024 — APRA CPS 220 Monitoring Dashboard\n'
             '(Gemini Pro Vision automated extraction + structured benchmarking)',
             fontsize=13, color=NAVY, y=1.01)
plt.savefig('apra_capital_dashboard.png', bbox_inches='tight', dpi=120)
plt.show()
print('Dashboard saved to apra_capital_dashboard.png')

---
## Week 3 Summary

| § | Topic | Key result |
|---|---|---|
| §1 | LSTM from scratch | Forget gate collapses at CPI drop; input gate spikes at commodity shock |
| §2 | Bitcoin LSTM (PyTorch) | 2-layer LSTM, MinMax scaling, 30-day lookback, RMSE reported |
| §3 | Positional encoding | cosine(pos5,pos6)≈0.97; cosine(pos5,pos55)≈0.14 — proximity encoded geometrically |
| §4 | Multi-head attention | Head 1: unless→§12.1 cross-reference; Head 2: syntactic structure |
| §5 | Vision Transformer (ViT) | 64 patches × 32-dim embeddings; [CLASS] attention map localises table cells |
| §6 | CLIP cross-modal | Similarity matrix diagonal dominates; zero-shot classification accuracy reported |
| §7 | BLIP-2 Q-Former | K=8 cross-attn; p_visual (K×D_LLM); domain fine-tune cost 2100→6 GPU-days |
| §8 | BloombergGPT | 708B tokens (51.27% FinPile); outperforms 176B BLOOM on finance benchmarks |
| §9 | LoRA efficiency | r=8: 64:1 compression, 1.56% of matrix; AUSTRAC Q&A 61.3%→84.7% |
| §10 | MoE routing | Top-2 of 8: 82% cost reduction; expert specialisation by token type |
| §11 | Gemini + VQA | ANZ CET1 12.30% PASS, credit provision FLAG; 4 financial VQA scenarios |
| §12 | APRA dashboard | Capital adequacy monitoring, multimodal failure modes, regulatory governance |

**Key Week 3 insights:**
- The transformer replaces LSTM's sequential bottleneck with parallel self-attention, enabling
  long-range cross-reference resolution in regulatory documents in a single forward pass
- ViT, CLIP, and BLIP-2 form a modular multimodal stack: ViT tokenises images as patch sequences;
  CLIP aligns modalities in a shared embedding space; BLIP-2's Q-Former bridges ViT and LLM at
  1.6% of joint-training compute — making domain adaptation accessible to APRA-regulated ADIs
- BloombergGPT demonstrates mixed domain + general training Pareto-dominates pure strategies;
  LoRA makes adaptation accessible without Bloomberg-scale compute
- VQA is powerful but must be embedded in GDPR Art. 22 / APRA APS 220 human-oversight workflows
  before any output influences a credit, compliance, or trading decision

**Week 4 preview:** Agentic AI — autonomous pipelines orchestrating LLMs, tools, and the multimodal
architectures from this week across multi-step financial workflows.

---

## References

Dosovitskiy, A., Beyer, L., Kolesnikov, A. et al. (2021). An image is worth 16x16 words. *ICLR 2021*.

Fedus, W., Zoph, B., & Shazeer, N. (2022). Switch Transformers. *JMLR, 23*(1), 1–39.

Google DeepMind. (2023). *Gemini: A family of highly capable multimodal models*. Technical Report.

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780.

Hric, J., & Lin, Y. (2026). *FINS5557 Applied AI in Finance*. UNSW Business School.

Hu, E., Shen, Y., Wallis, P. et al. (2021). LoRA: Low-rank adaptation of large language models. *ICLR 2022*.

Li, J., Li, D., Savarese, S., & Hoi, S. (2023). BLIP-2: Bootstrapping language-image pre-training with frozen image encoders and large language models. *ICML 2023*, 19730–19742.

Mistral AI. (2023). *Mixtral of experts*. Technical Report.

Radford, A., Narasimhan, K., Salimans, T., & Sutskever, I. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.

Radford, A., Kim, J.W., Hallacy, C. et al. (2021). Learning transferable visual models from natural language supervision. *ICML 2021*, 8748–8763.

Vaswani, A., Shazeer, N., Parmar, N. et al. (2017). Attention is all you need. *NeurIPS 30*.

Wu, S., Irsoy, O., Lu, S. et al. (2023). BloombergGPT: A large language model for finance. *arXiv:2303.17564*.